## Ajout d’un lieu manuel aux triptyques

Cette cellule ajoute un lieu à chaque triptyque à partir des annotations manuelles du fichier `.entities`.

On considère comme lieu manuel :

- les entités annotées `FAC`, `LOC`, `GPE` ou `VEH` ;
- toutes les entités qui ont une annotation `SPACESEM`, même si leur type n’est pas un type de lieu.

Cela permet de récupérer les lieux annotés comme :

- `CAD cadre` : lieu où se déroule l’action ;
- `SRC source` : point de départ ;
- `GOL goal` : point d’arrivée ;
- `MED median` : lieu de passage.

Pour relier un lieu à un triptyque, on utilise d’abord la phrase :

`Num_paragr` + `Num_phrase` côté triptyques  
`paragraph_ID` + `sentence_ID` côté annotations manuelles.

S’il y a plusieurs lieux dans une même phrase, la cellule choisit en priorité :

1. le lieu qui contient le token de l’objet du triptyque ;
2. sinon le lieu qui contient le token du sujet ;
3. sinon le lieu annoté comme cadre, but, source ou médian dans la phrase.

Le résultat est un fichier de triptyques enrichi avec :

- `lieu_manuel`
- `lieu_mention_manuelle`
- `relation_lieu_manuelle`
- `type_lieu_manuel`
- `tous_lieux_manuels`

In [4]:
import pandas as pd

# Chemins à adapter
TRIPTYQUES_CSV = "../results/csv_triptyques/manuel_chap1to5_chap_temps.csv"
ENTITIES_CSV = "../data/SACR/all_annots.sacr.entities"
OUT_CSV = "../results/csv_triptyques/manuel_chap1to5_chap_temps_lieux_manuels.csv"

# Chargement des triptyques et des annotations manuelles
trip = pd.read_csv(TRIPTYQUES_CSV)
entities = pd.read_csv(ENTITIES_CSV, sep="\t")

# Types d'entités qui correspondent directement à des lieux
PLACE_TYPES = ["FAC", "LOC", "GPE", "VEH"]

# Priorité si plusieurs lieux sont présents dans la même phrase
ROLE_PRIORITY = {
    "CAD cadre": 1,
    "GOL goal": 2,
    "SRC source": 3,
    "MED median": 4,
}

# Garde les lieux annotés manuellement
# On garde aussi les entités avec SPACESEM, même si leur cat n'est pas FAC/LOC/GPE/VEH
lieux = entities[
    (
        entities["cat"].isin(PLACE_TYPES)
        & ~entities["POS_tag"].isin(["VERB", "AUX"])
        & (entities["FUNCT"] != "VERB verb")
    )
    | entities["SPACESEM"].notna()
].copy()

# On enlève les temps, au cas où
lieux = lieux[lieux["cat"] != "TIME"].copy()

# Nom stable du lieu : on privilégie la chaîne de coréférence manuelle
lieux["lieu_manuel"] = lieux["COREF_name"].fillna(lieux["text"])

# Mention exacte dans le texte
lieux["lieu_mention_manuelle"] = lieux["text"]

# Rôle spatial manuel : CAD, SRC, GOL, MED
lieux["relation_lieu_manuelle"] = lieux["SPACESEM"].fillna("lieu_sans_relation")

# Type manuel : FAC, LOC, GPE, VEH, etc.
lieux["type_lieu_manuel"] = lieux["cat"]

# Score de priorité du rôle spatial
lieux["priorite_role"] = lieux["SPACESEM"].map(ROLE_PRIORITY).fillna(5)

# Regroupe les lieux par phrase
lieux_par_phrase = {
    key: group.sort_values(["priorite_role", "start_token"]).copy()
    for key, group in lieux.groupby(["paragraph_ID", "sentence_ID"])
}

# Vérifie si un token du triptyque tombe dans une mention de lieu
def token_dans_lieu(token_id, lieu):
    if pd.isna(token_id):
        return False

    token_id = int(token_id)

    return int(lieu["start_token"]) <= token_id <= int(lieu["end_token"])

# Colle les valeurs uniques dans une seule cellule
def valeurs_uniques(serie):
    valeurs = []

    for x in serie:
        if pd.isna(x):
            continue

        x = str(x)

        if x not in valeurs:
            valeurs.append(x)

    if len(valeurs) == 0:
        return pd.NA

    return " ; ".join(valeurs)

# Choisit le meilleur lieu pour un triptyque
def choisir_lieu(row):
    key = (row["Num_paragr"], row["Num_phrase"])

    # Aucun lieu manuel dans cette phrase
    if key not in lieux_par_phrase:
        return pd.Series({
            "lieu_manuel": pd.NA,
            "lieu_mention_manuelle": pd.NA,
            "relation_lieu_manuelle": pd.NA,
            "type_lieu_manuel": pd.NA,
            "tous_lieux_manuels": pd.NA,
        })

    candidats = lieux_par_phrase[key].copy()

    # Priorité 0 : le lieu contient l'objet du triptyque
    # Priorité 1 : le lieu contient le sujet du triptyque
    # Priorité 2 : lieu seulement présent dans la même phrase
    scores = []

    for _, lieu in candidats.iterrows():
        if token_dans_lieu(row["DocID_objet"], lieu):
            scores.append(0)
        elif token_dans_lieu(row["DocID_sujet"], lieu):
            scores.append(1)
        else:
            scores.append(2)

    candidats["priorite_match"] = scores

    # Choisit le meilleur candidat
    candidats = candidats.sort_values(
        ["priorite_match", "priorite_role", "start_token"]
    )

    principal = candidats.iloc[0]

    return pd.Series({
        "lieu_manuel": principal["lieu_manuel"],
        "lieu_mention_manuelle": principal["lieu_mention_manuelle"],
        "relation_lieu_manuelle": principal["relation_lieu_manuelle"],
        "type_lieu_manuel": principal["type_lieu_manuel"],
        "tous_lieux_manuels": valeurs_uniques(candidats["lieu_manuel"]),
    })

# Ajoute les colonnes de lieux manuels aux triptyques
lieux_triptyques = trip.apply(choisir_lieu, axis=1)

trip_lieux = pd.concat([trip, lieux_triptyques], axis=1)

# Exporte le fichier enrichi
trip_lieux.to_csv(OUT_CSV, index=False, encoding="utf-8")

print("Triptyques enrichis :", trip_lieux.shape)
print("Triptyques avec lieu manuel :", trip_lieux["lieu_manuel"].notna().sum())
print("CSV écrit :", OUT_CSV)

trip_lieux[[
    "Phrase",
    "Sujet",
    "Verbe",
    "Objet",
    "lieu_manuel",
    "lieu_mention_manuelle",
    "relation_lieu_manuelle",
    "type_lieu_manuel",
    "tous_lieux_manuels"
]].head(30)

Triptyques enrichis : (1255, 52)
Triptyques avec lieu manuel : 547
CSV écrit : ../results/csv_triptyques/manuel_chap1to5_chap_temps_lieux_manuels.csv


,Phrase,Sujet,Verbe,Objet,lieu_manuel,lieu_mention_manuelle,relation_lieu_manuelle,type_lieu_manuel,tous_lieux_manuels
0,"En l' année 1872, la maison portant le numéro ...",NaN,portant,le numéro 7 de Saville-row Burlington Gardens-...,maison_fogg,la maison portant le numéro 7 de saville - row,lieu_sans_relation,FAC,maison_fogg ; ville_londres ; rue_fogg ; refor...
1,"En l' année 1872, la maison portant le numéro ...",Sheridan,mourut,en 1814,ville_londres,londres,CAD cadre,GPE,ville_londres ; maison_fogg ; rue_fogg ; refor...
2,"En l' année 1872, la maison portant le numéro ...",la maison,habitée,En l'année 1872,maison_fogg,la maison portant le numéro 7 de saville - row,lieu_sans_relation,FAC,maison_fogg ; ville_londres ; rue_fogg ; refor...
3,"En l' année 1872, la maison portant le numéro ...",la maison,habitée,par Phileas Fogg esq.,maison_fogg,la maison portant le numéro 7 de saville - row,lieu_sans_relation,FAC,maison_fogg ; ville_londres ; rue_fogg ; refor...
4,"En l' année 1872, la maison portant le numéro ...",la maison,habitée,l'un des membres les plus singuliers et les pl...,maison_fogg,la maison portant le numéro 7 de saville - row,lieu_sans_relation,FAC,maison_fogg ; ville_londres ; rue_fogg ; refor...
5,"En l' année 1872, la maison portant le numéro ...",la maison,semblât,NaN,maison_fogg,la maison portant le numéro 7 de saville - row,lieu_sans_relation,FAC,maison_fogg ; ville_londres ; rue_fogg ; refor...
6,"En l' année 1872, la maison portant le numéro ...",NaN,prendre,à tâche,ville_londres,londres,CAD cadre,GPE,ville_londres ; maison_fogg ; rue_fogg ; refor...
7,"En l' année 1872, la maison portant le numéro ...",NaN,faire,NaN,ville_londres,londres,CAD cadre,GPE,ville_londres ; maison_fogg ; rue_fogg ; refor...
8,"En l' année 1872, la maison portant le numéro ...",qui,pût,NaN,ville_londres,londres,CAD cadre,GPE,ville_londres ; maison_fogg ; rue_fogg ; refor...
9,"En l' année 1872, la maison portant le numéro ...",qui,attirer,l'attention,ville_londres,londres,CAD cadre,GPE,ville_londres ; maison_fogg ; rue_fogg ; refor...


In [5]:
trip_lieux[trip_lieux["relation_lieu_manuelle"] == "MED median"]

,Phrase,Sujet,Verbe,Objet,Dep_sujet,Dep_verbe,Dep_objet,ID_sujet,ID_verbe,ID_objet,...,time_duration_value,time_duration_unit,time_duration_relation,time_source,time_ud_governor,lieu_manuel,lieu_mention_manuelle,relation_lieu_manuelle,type_lieu_manuel,tous_lieux_manuels
87,Ceux qui avaient l' honneur de le connaître un...,Ceux,est,sur ce chemin direct,nsubj,ccomp,obl:mod,1.0,21,24.0,...,NaN,NaN,NaN,precedent_context,habitée,M1,ce chemin direct,MED median,FAC,M1 ; reform_club ; maison_fogg
375,Il avait fait dix maisons.,Il,fait,dix maisons,nsubj,root,obj,1.0,3,5.0,...,NaN,NaN,NaN,precedent_context,cherché,noRef_1103,dix maisons,MED median,FAC,noRef_1103
1075,"Mais le train n' avait pas dépassé Sydenham, q...",le train,dépassé,Sydenham,nsubj,conj,obj,3.0,7,8.0,...,NaN,NaN,NaN,precedent_context,retentit,ville_sydenham,sydenham,MED median,GPE,ville_sydenham ; train_paris
1076,"Mais le train n' avait pas dépassé Sydenham, q...",Passepartout,poussait,un véritable cri de désespoir,nsubj,ccomp,obj,11.0,12,15.0,...,NaN,NaN,NaN,precedent_context,retentit,ville_sydenham,sydenham,MED median,GPE,ville_sydenham ; train_paris


In [6]:
trip_lieux[trip_lieux["relation_lieu_manuelle"] == "SRC source"]

,Phrase,Sujet,Verbe,Objet,Dep_sujet,Dep_verbe,Dep_objet,ID_sujet,ID_verbe,ID_objet,...,time_duration_value,time_duration_unit,time_duration_relation,time_source,time_ud_governor,lieu_manuel,lieu_mention_manuelle,relation_lieu_manuelle,type_lieu_manuel,tous_lieux_manuels
90,Ceux qui avaient l' honneur de le connaître un...,il,venir,de sa maison au club,nsubj,advcl,obl:arg,27.0,32,35.0,...,NaN,NaN,NaN,precedent_context,habitée,maison_fogg,de sa maison,SRC source,FAC,maison_fogg ; reform_club ; M1
189,"À onze heures et demie sonnant, Mr. Fogg devai...",Mr. Fogg,quitter,la maison,nsubj,xcomp,obj,8.0,17,19.0,...,NaN,NaN,NaN,ud_scope_anchor,devait,maison_fogg,la maison,SRC source,FAC,maison_fogg ; reform_club
223,Mais voilà cinq ans que j' ai quitté la France...,j',quitté,la France,nsubj,ccomp,obj,6.0,8,10.0,...,NaN,NaN,NaN,precedent_context,devait,pays_france,la france,SRC source,GPE,pays_france ; pays_angleterre ; M35
408,Il la parcourut de la cave au grenier.,Il,parcourut,de la cave au grenier,nsubj,root,obl:arg,1.0,3,6.0,...,NaN,NaN,NaN,precedent_context,cherché,M12,de la cave,SRC source,_,M12 ; maison_fogg ; M13
440,"Il comprenait- depuis huit heures du matin, he...",il,quittait,sa maison,nsubj,acl:relcl,obj,29.0,30,32.0,...,NaN,NaN,NaN,ud_scope_anchor,comprenait,maison_fogg,sa maison,SRC source,FAC,maison_fogg ; reform_club
486,Phileas Fogg avait quitté sa maison de Saville...,Phileas Fogg,quitté,sa maison de Saville-row,nsubj,root,obj,1.0,4,6.0,...,NaN,NaN,NaN,ud_direct,quitté,maison_fogg,sa maison de saville - row,SRC source,FAC,maison_fogg ; rue_fogg ; reform_club
487,Phileas Fogg avait quitté sa maison de Saville...,Phileas Fogg,quitté,à onze heures et demie,nsubj,root,obl:mod,1.0,4,13.0,...,NaN,NaN,NaN,objet,quitté,maison_fogg,sa maison de saville - row,SRC source,FAC,maison_fogg ; rue_fogg ; reform_club
488,Phileas Fogg avait quitté sa maison de Saville...,il,placé,cinq cent soixante-quinze fois son pied droit,nsubj,advcl,obl:mod,46.0,21,25.0,...,NaN,NaN,NaN,ud_scope_anchor,quitté,maison_fogg,sa maison de saville - row,SRC source,FAC,maison_fogg ; rue_fogg ; reform_club
489,Phileas Fogg avait quitté sa maison de Saville...,il,placé,devant son pied gauche,nsubj,advcl,obl:arg,46.0,21,31.0,...,NaN,NaN,NaN,ud_scope_anchor,quitté,maison_fogg,sa maison de saville - row,SRC source,FAC,maison_fogg ; rue_fogg ; reform_club
490,Phileas Fogg avait quitté sa maison de Saville...,il,placé,devant son pied droit,nsubj,advcl,obl:arg,46.0,21,43.0,...,NaN,NaN,NaN,ud_scope_anchor,quitté,maison_fogg,sa maison de saville - row,SRC source,FAC,maison_fogg ; rue_fogg ; reform_club
